# 🧠 Ablation Study: Evaluasi LLM-as-Labeler (Llama-3-8B-Instruct) pada GPU T4
## Perluasan Dataset IndQNER Menggunakan Arsitektur Two-Tier Labeling

Notebook ini dirancang untuk dijalankan di **Google Colab (GPU T4)** guna melakukan eksperimen komparatif ilmiah:
1. **Eksperimen A (Validasi Akurasi LLM)**: Menguji performa **Llama-3-8B** (*Zero-Shot* vs *Few-Shot*) terhadap 237 entitas ground-truth resmi dari `indqner_1.0/test.txt`.
2. **Eksperimen B (Pelabelan Entitas Unmapped)**: Melabeli 33 entitas baru dari 114 surah yang tidak terdaftar di Kamus Lapis 1.

> **Panduan GPU**: Pastikan runtime Colab diatur ke **T4 GPU** (`Runtime` -> `Change runtime type` -> `T4 GPU`).

### 1. Instalasi Dependensi GPU T4

In [ ]:
!pip install -q transformers accelerate bitsandbytes torch scikit-learn

### 2. Memuat Model Llama-3-8B-Instruct (4-Bit Quantized via BitsAndBytes)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline

# Menggunakan model Llama-3-8B-Instruct 4-bit (sangat hemat VRAM ~5.5GB, pas di T4)
model_id = "unsloth/llama-3-8b-Instruct-bnb-4bit"

print("⏳ Memuat tokenizer dan model Llama-3-8B ke GPU T4...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16
)
print("✅ Model Llama-3-8B berhasil dimuat di GPU!")

### 3. Unggah Berkas Payload `llm_colab_payload.json`

In [ ]:
import json, os
from google.colab import files

if not os.path.exists("llm_colab_payload.json"):
    print("Silakan upload berkas 'llm_colab_payload.json' dari folder lokal data/intermediate/:")
    uploaded = files.upload()

with open("llm_colab_payload.json", "r", encoding="utf-8") as f:
    payload_data = json.load(f)

print(f"✅ Metadata Payload: {payload_data['metadata']}")
print(f"• Jumlah Sampel Benchmark Ground-Truth: {len(payload_data['benchmark_entities'])}")
print(f"• Jumlah Entitas Unmapped 114 Surah: {len(payload_data['unmapped_entities'])}")

### 4. Definisi 20 Kategori Kelas Resmi & Prompt Builder

In [ ]:
VALID_CLASSES = [
    "Allah", "Messenger", "Prophet", "Person", "HolyBook",
    "AstronomicalBody", "Event", "AfterlifeLocation", "GeographicalLocation", "Artifact",
    "Throne", "Religion", "Angel", "Color", "Language",
    "Food", "Fruit", "FalseDeity", "Sentient", "TheBookOfAllah"
]

FEW_SHOT_EXEMPLARS = [
    {"s": "Kitab (Al-Qur’an) ini tidak ada keraguan di dalamnya.", "e": "Al-Qur’an", "c": "HolyBook"},
    {"s": "Katakanlah, 'Kami beriman kepada Allah dan apa yang diturunkan kepada Ibrahim dan Musa.'", "e": "Musa", "c": "Messenger"},
    {"s": "Dia mengajarkan kepada Adam nama-nama benda seluruhnya.", "e": "Adam", "c": "Prophet"},
    {"s": "Dia menjadikan bagimu bumi sebagai hamparan dan langit sebagai atap.", "e": "bumi", "c": "AstronomicalBody"},
    {"s": "Sesungguhnya orang-orang yang beriman, orang-orang Yahudi, dan Nasrani.", "e": "Nasrani", "c": "Religion"},
    {"s": "Dan ingatlah ketika Kami selamatkan kamu dari pengikut-pengikut Fir‘aun.", "e": "Fir‘aun", "c": "Person"}
]

def construct_prompt(sentence: str, entity: str, mode: str = "few_shot") -> str:
    classes_list_str = "\n".join([f"- {c}" for c in VALID_CLASSES])
    sys_msg = (
        "Anda adalah sistem anotasi Named Entity Recognition (NER) khusus teks Terjemahan Al-Qur'an Kementerian Agama RI.\n"
        "Tugas: Klasifikasikan entitas target ke dalam TEPAT SATU dari 20 kelas berikut:\n"
        f"{classes_list_str}\n\n"
        "ATURAN KETAT:\n"
        "1. Jawab HANYA DENGAN SATU NAMA KELAS persis sesuai daftar di atas.\n"
        "2. DILARANG menambah penjelasan, pengantar, atau tanda kutip."
    )
    
    examples_str = ""
    if mode == "few_shot":
        examples_str = "\nContoh Acuan:\n"
        for ex in FEW_SHOT_EXEMPLARS:
            examples_str += f"Kalimat: \"{ex['s']}\" | Entitas: \"{ex['e']}\" -> Kelas: {ex['c']}\n"
            
    user_msg = (
        f"{examples_str}\n"
        f"Kalimat: \"{sentence}\"\n"
        f"Entitas: \"{entity}\"\n"
        "Kelas:"
    )
    
    messages = [
        {"role": "system", "content": sys_msg},
        {"role": "user", "content": user_msg}
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def parse_class_response(raw_text: str) -> str:
    # Pembersihan output untuk mengekstrak 1 kelas sah
    clean = raw_text.strip().replace("*", "").replace("\"", "").replace("'", "")
    for cls in VALID_CLASSES:
        if cls.lower() in clean.lower():
            return cls
    return "Person" # Fallback paling netral

### 5. Fungsi Eksekusi Inferensi Batch di GPU T4

In [ ]:
def run_inference(items, mode="few_shot"):
    results = []
    print(f"🚀 Menjalankan inferensi untuk {len(items)} entitas (Mode: {mode})...")
    
    for i, it in enumerate(items):
        prompt = construct_prompt(it["sentence"], it["entity"], mode=mode)
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=10,
                temperature=0.01,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )
            
        gen_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
        pred_class = parse_class_response(gen_text)
        
        res = dict(it)
        res["predicted_class"] = pred_class
        res["raw_llm_output"] = gen_text.strip()
        res["mode"] = mode
        results.append(res)
        
        if (i + 1) % 50 == 0 or (i + 1) == len(items):
            print(f"  [{i+1}/{len(items)}] Terproses...")
            
    return results

### 6. Eksperimen A: Evaluasi Kuantitatif pada Benchmark Test Set (237 Entitas Ground-Truth)
Mengukur akurasi ilmiah Llama-3-8B untuk dimasukkan ke Bab IV Skripsi.

In [ ]:
from sklearn.metrics import classification_report, accuracy_score

benchmark_items = payload_data["benchmark_entities"]

# Jalankan Few-Shot
preds_few_shot = run_inference(benchmark_items, mode="few_shot")

y_true = [it["ground_truth_class"] for it in preds_few_shot]
y_pred = [it["predicted_class"] for it in preds_few_shot]

acc = accuracy_score(y_true, y_pred)
print("\n" + "="*60)
print(f"HASIL AKURASI LLAMA-3-8B FEW-SHOT PADA TEST SET: {acc*100:.2f}%")
print("="*60)
print(classification_report(y_true, y_pred, zero_division=0))

### 7. Eksperimen B: Pelabelan 33 Entitas Unmapped Korpus 114 Surah

In [ ]:
unmapped_items = payload_data["unmapped_entities"]

preds_unmapped = run_inference(unmapped_items, mode="few_shot")

print("\nHASIL PELABELAN 33 ENTITAS UNMAPPED OLEH LLAMA-3-8B:")
for it in preds_unmapped:
    print(f"• QS {it['surah']}:{it['ayah']} | Entitas: '{it['entity']}' -> Prediksi: {it['predicted_class']}")

### 8. Unduh Berkas Hasil Prediksi (`llama3_labeling_results.json`)

In [ ]:
output_data = {
    "metadata": {
        "model": "meta-llama/Meta-Llama-3-8B-Instruct (4-bit)",
        "hardware": "Google Colab GPU T4",
        "benchmark_accuracy": float(acc)
    },
    "benchmark_results": preds_few_shot,
    "unmapped_results": preds_unmapped
}

with open("llama3_labeling_results.json", "w", encoding="utf-8") as f:
    json.dump(output_data, f, ensure_ascii=False, indent=2)

print("💾 Menyimpan berkas hasil prediksi...")
files.download("llama3_labeling_results.json")
print("✅ Berkas 'llama3_labeling_results.json' siap diunduh dan dimasukkan kembali ke proyek!")